# Análise de viabilidade (`labdados.analise_viabilidade`)

<a href="https://colab.research.google.com/github/lab-dados/labdados-sdk/blob/main/examples/notebooks/analise_viabilidade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Avalia o tamanho de um recorte **antes** de fazer raspagem. Para cada combinação de tribunal + filtros, conta:

- **`datajud`** — quantos processos existem (1ª e 2ª instância, via API CNJ).
- **`jurisprudencia`** — quantos acórdãos a busca textual retorna.
- **`sentencas`** — quantas sentenças (1ª instância — disponível só em alguns tribunais).

Devolve um veredito (`viable` / `caveats` / `unviable`) e um relatório em PDF.

📌 **Sempre roda local** — não tem modo nuvem nem precisa de API key.

Cada célula de execução é independente: rode só o cenário (datajud / jurisprudência / sentenças) que te interessa. As células de instalação são pré-requisitos compartilhados.

📘 Doc completa: <https://lab-dados.github.io/labdados-sdk>

## 1. Instalar

O extra `[viabilidade]` traz juscraper + pydantic + jinja2. Para gerar o PDF do relatório, instalamos também o Quarto + Typst (binários do sistema).

In [ ]:
%pip install -q 'labdados[viabilidade]'

In [ ]:
# Quarto + Typst — opcional, só pra ter o PDF.
# Sem isso, o relatório sai em markdown.
!wget -q -O quarto.deb https://github.com/quarto-dev/quarto-cli/releases/download/v1.5.57/quarto-1.5.57-linux-amd64.deb
!dpkg -i quarto.deb > /dev/null 2>&1 || apt-get install -fy -qq
!quarto --version

## 2. Datajud com filtro por assunto

Exemplo: ações sobre planos de saúde (medicamentos + home care) em TJSP/TJRJ/TJMG entre 2020 e 2024.

Os códigos `assuntos_cnj` vêm da TPU do CNJ (lista completa em [abjur/tpur](https://github.com/abjur/tpur)).

In [ ]:
import labdados

ana = labdados.analise_viabilidade(
    descricao='Ações contra planos de saúde — TJSP, TJRJ, TJMG (2020-2024)',
    listagem='datajud',
    tribunais=['tjsp', 'tjrj', 'tjmg'],
    assuntos_cnj=[
        '11884',  # Fornecimento de Medicamentos
        '14759',  # Tratamento Domiciliar (Home Care)
    ],
    inicio='2020-01-01',
    fim='2024-12-31',
    saida='relatorios/',
    notas='Pesquisa exploratória para tese de doutorado em direito civil.',
)

print('Veredito:        ', ana['results']['verdict'])
print('Total aproximado:', ana['results'].get('total_aproximado'))
print('PDF:             ', ana.get('report_pdf'))
print('Markdown:        ', ana.get('report_md'))
print()
print('--- Pontos de atenção ---')
for h in ana['results'].get('highlights', []):
    print('•', h)

## 3. Jurisprudência (busca textual em acórdãos)

In [ ]:
import labdados

ana = labdados.analise_viabilidade(
    descricao='Acórdãos sobre nepotismo no STF e STJ',
    listagem='jurisprudencia',
    tribunais=['stf', 'stj'],
    palavras_chave='nepotismo',
    saida='relatorios/',
)
ana['results']

## 4. Sentenças (1º grau, disponibilidade limitada)

Hoje só TJSP, TJES e alguns outros expõem busca pública de sentenças.

In [ ]:
import labdados

ana = labdados.analise_viabilidade(
    descricao='Sentenças do TJSP citando "home office" em direito do trabalho',
    listagem='sentencas',
    tribunais=['tjsp'],
    palavras_chave='"home office" "vínculo de emprego"',
    saida='relatorios/',
)
ana['results']

## Como interpretar o veredito

| Veredito   | Significado                                                        |
|------------|--------------------------------------------------------------------|
| `viable`   | Volume gerenciável (Datajud: até 50 mil) e sem erros — pode coletar |
| `caveats`  | Volume alto **ou** erros parciais — fatie em sub-recortes           |
| `unviable` | Sem resultados ou todos os tribunais falharam — revise filtros      |

## Onde achar códigos CNJ

- [SGT do CNJ](https://www.cnj.jus.br/sgt/consulta_publica_assuntos.php) — busca interativa.
- [abjur/tpur](https://github.com/abjur/tpur) — histórico completo em CSV.

Doc completa dos parâmetros: <https://lab-dados.github.io/labdados-sdk/api/analise_viabilidade.html>